In [5]:
import pandas as pd
import numpy as np

In [6]:
file_path = "38个浮游植物样品数据-2025.12样品.xlsx"

df = pd.read_excel(file_path)

print(df.head())
print(df.columns.tolist())

     样品编号  浓缩体积（mL）  采样体积（L）  计数体积（mL）      中文名  \
0  PT-4 B        40        2         1    铁氏束毛藻   
1  PT-4 B        40        2         1     圆筛藻属   
2  PT-4 B        40        2         1    密聚角毛藻   
3  PT-4 B        40        2         1  具边线性圆筛藻   
4  PT-4 B        40        2         1    海洋角毛藻   

                                拉丁名  类群  个体数量  
0          Trichodesmium thiebautii  蓝藻     1  
1                 Coscinodiscus sp.  硅藻     5  
2            Chaetoceros coarctatus  硅藻     2  
3  Coscinodiscus marginato-lineatus  硅藻     2  
4             Chaetoceros pelagicus  硅藻     2  
['样品编号', '浓缩体积（mL）', '采样体积（L）', '计数体积（mL）', '中文名', '拉丁名', '类群', '个体数量']


In [7]:
df.columns = df.columns.astype(str).str.strip()

In [8]:
numeric_columns = [
    "浓缩体积（mL）",
    "采样体积（L）",
    "计数体积（mL）",
    "个体数量"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [9]:
print(df.columns.tolist())

['样品编号', '浓缩体积（mL）', '采样体积（L）', '计数体积（mL）', '中文名', '拉丁名', '类群', '个体数量']


In [10]:
print(df["类群"].value_counts())

类群
硅藻    505
甲藻     52
蓝藻     10
绿藻      4
黄藻      1
Name: count, dtype: int64


In [11]:
df["丰度_cells_L"] = (
    df["个体数量"]
    * df["浓缩体积（mL）"]
    / (
        df["计数体积（mL）"]
        * df["采样体积（L）"]
    )
)

In [12]:
print(
    df[
        [
            "样品编号",
            "拉丁名",
            "类群",
            "个体数量",
            "丰度_cells_L"
        ]
    ].head(20)
)

      样品编号                                        拉丁名  类群  个体数量  丰度_cells_L
0   PT-4 B                   Trichodesmium thiebautii  蓝藻     1        20.0
1   PT-4 B                          Coscinodiscus sp.  硅藻     5       100.0
2   PT-4 B                     Chaetoceros coarctatus  硅藻     2        40.0
3   PT-4 B           Coscinodiscus marginato-lineatus  硅藻     2        40.0
4   PT-4 B                      Chaetoceros pelagicus  硅藻     2        40.0
5   PT-4 B  Rhizosolenia styliformis var. longispina   硅藻     1        20.0
6   PT-4 B                   Rhizosolenia styliformis  硅藻     1        20.0
7   PT-4 B                             Cyclotella sp.  硅藻     1        20.0
8   PT-4 B                      Rhizosolenia setigera  硅藻     1        20.0
9   PT-4 B                  Thalassiothrix longissima  硅藻     2        40.0
10  PT-4 B                  Protoperidinium pyriforme  甲藻     1        20.0
11  PT-4 B                              Nitzschia sp.  硅藻    22       440.0
12  PT-4 B  

In [14]:
print(df["丰度_cells_L"].describe())

count     572.000000
mean       85.314685
std       137.889035
min        20.000000
25%        20.000000
50%        40.000000
75%        80.000000
max      1420.000000
Name: 丰度_cells_L, dtype: float64


In [15]:
print(
    df[
        ~np.isfinite(df["丰度_cells_L"])
    ]
)

Empty DataFrame
Columns: [样品编号, 浓缩体积（mL）, 采样体积（L）, 计数体积（mL）, 中文名, 拉丁名, 类群, 个体数量, 丰度_cells_L]
Index: []


In [16]:
species_sample = (
    df.groupby(
        [
            "样品编号",
            "中文名",
            "拉丁名",
            "类群"
        ],
        as_index=False
    )
    .agg(
        个体数量=("个体数量", "sum"),
        丰度_cells_L=("丰度_cells_L", "sum")
    )
)

In [17]:
print(species_sample.head())
print(species_sample.shape)

              样品编号    中文名                          拉丁名  类群  个体数量  丰度_cells_L
0  HW25110501001-1   三叉角藻         Ceratium trichoceors  甲藻     3        60.0
1  HW25110501001-1  哈德半盘藻     Hemidiscus hardmannianus  硅藻    12       240.0
2  HW25110501001-1  布氏双尾藻         Ditylum brightwellii  硅藻     2        40.0
3  HW25110501001-1  星脐圆筛藻  Coscinodiscus asteromphalus  硅藻    10       200.0
4  HW25110501001-1  格氏圆筛藻         Coscinodiscus granii  硅藻     1        20.0
(572, 6)


In [18]:
target_groups = ["硅藻", "甲藻", "蓝藻"]

three_groups = species_sample[
    species_sample["类群"].isin(target_groups)
].copy()

In [19]:
group_abundance_long = (
    three_groups.groupby(
        ["样品编号", "类群"],
        as_index=False
    )["丰度_cells_L"]
    .sum()
)

In [20]:
group_abundance = (
    group_abundance_long.pivot(
        index="样品编号",
        columns="类群",
        values="丰度_cells_L"
    )
    .fillna(0)
    .reset_index()
)

In [21]:
for group in target_groups:
    if group not in group_abundance.columns:
        group_abundance[group] = 0

In [22]:
group_abundance = group_abundance[
    ["样品编号", "硅藻", "甲藻", "蓝藻"]
]

In [23]:
group_abundance = group_abundance.rename(
    columns={
        "硅藻": "硅藻丰度_cells_L",
        "甲藻": "甲藻丰度_cells_L",
        "蓝藻": "蓝藻丰度_cells_L"
    }
)

In [24]:
total_abundance = (
    species_sample.groupby(
        "样品编号",
        as_index=False
    )["丰度_cells_L"]
    .sum()
    .rename(
        columns={
            "丰度_cells_L": "浮游植物总丰度_cells_L"
        }
    )
)

In [25]:
sample_summary = group_abundance.merge(
    total_abundance,
    on="样品编号",
    how="left"
)

In [26]:
sample_summary["其他类群丰度_cells_L"] = (
    sample_summary["浮游植物总丰度_cells_L"]
    - sample_summary["硅藻丰度_cells_L"]
    - sample_summary["甲藻丰度_cells_L"]
    - sample_summary["蓝藻丰度_cells_L"]
)

In [27]:
sample_summary["硅藻相对丰度"] = (
    sample_summary["硅藻丰度_cells_L"]
    / sample_summary["浮游植物总丰度_cells_L"]
)

sample_summary["甲藻相对丰度"] = (
    sample_summary["甲藻丰度_cells_L"]
    / sample_summary["浮游植物总丰度_cells_L"]
)

sample_summary["蓝藻相对丰度"] = (
    sample_summary["蓝藻丰度_cells_L"]
    / sample_summary["浮游植物总丰度_cells_L"]
)

In [28]:
relative_columns = [
    "硅藻相对丰度",
    "甲藻相对丰度",
    "蓝藻相对丰度"
]

sample_summary[relative_columns] = (
    sample_summary[relative_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [29]:
print(sample_summary.head())

              样品编号  硅藻丰度_cells_L  甲藻丰度_cells_L  蓝藻丰度_cells_L  浮游植物总丰度_cells_L  \
0  HW25110501001-1         620.0          60.0        1420.0           2100.0   
1  HW25110501002-1         560.0          80.0         200.0            840.0   
2  HW25110501003-1        1000.0         100.0        1080.0           2180.0   
3  HW25110501004-1        1980.0          80.0        1360.0           3420.0   
4           PT-1 B        1800.0          20.0           0.0           1820.0   

   其他类群丰度_cells_L    硅藻相对丰度    甲藻相对丰度    蓝藻相对丰度  
0             0.0  0.295238  0.028571  0.676190  
1             0.0  0.666667  0.095238  0.238095  
2             0.0  0.458716  0.045872  0.495413  
3             0.0  0.578947  0.023392  0.397661  
4             0.0  0.989011  0.010989  0.000000  


In [30]:
total_samples = species_sample["样品编号"].nunique()

print("总样品数：", total_samples)

总样品数： 42


In [31]:
sample_ids = sorted(species_sample["样品编号"].dropna().unique())

for sample_id in sample_ids:
    print(repr(sample_id))

'HW25110501001-1'
'HW25110501002-1'
'HW25110501003-1'
'HW25110501004-1'
'PT-1 B'
'PT-1 D'
'PT-2 B'
'PT-2 D'
'PT-3 B'
'PT-3 D'
'PT-4 B'
'PT-4 D'
'PT-6 B'
'PT-6 D'
'SG-1'
'SG-2'
'SG-3'
'SG-4'
'XS-1 B'
'XS-1 D'
'XS-10 B'
'XS-10 D'
'XS-11 B'
'XS-11 D'
'XS-12 B'
'XS-12 D'
'XS-2 B'
'XS-2 D'
'XS-3 B'
'XS-3 D'
'XS-4 B'
'XS-4 D'
'XS-5 B'
'XS-5 D'
'XS-6 B'
'XS-6 D'
'XS-7 B'
'XS-7 D'
'XS-8 B'
'XS-8 D'
'XS-9 B'
'XS-9 D'


In [32]:
species_summary = (
    species_sample.groupby(
        ["中文名", "拉丁名", "类群"],
        as_index=False
    )
    .agg(
        物种总丰度_n_i=("丰度_cells_L", "sum"),
        出现样品数=("样品编号", "nunique")
    )
)

In [33]:
N = species_summary["物种总丰度_n_i"].sum()

print("全部样品全部物种累计丰度 N：", N)

全部样品全部物种累计丰度 N： 48800.0


In [35]:
species_summary["出现频率_f_i"] = (
    species_summary["出现样品数"] / total_samples
)

In [36]:
species_summary["相对丰度_n_i_N"] = (
    species_summary["物种总丰度_n_i"] / N
)

species_summary["优势度_Y"] = (
    species_summary["相对丰度_n_i_N"]
    * species_summary["出现频率_f_i"]
)

In [37]:
species_summary["是否优势种"] = np.where(
    species_summary["优势度_Y"] > 0.02,
    "优势种",
    "非优势种"
)

In [38]:
species_summary = species_summary.sort_values(
    "优势度_Y",
    ascending=False
).reset_index(drop=True)

In [39]:
dominant_species = species_summary[
    species_summary["优势度_Y"] > 0.02
].copy()

print(dominant_species)

     中文名                  拉丁名  类群  物种总丰度_n_i  出现样品数  出现频率_f_i  相对丰度_n_i_N  \
0   菱形藻属        Nitzschia sp.  硅藻    10340.0     37  0.880952    0.211885   
1   舟形藻属         Navicula sp.  硅藻     5780.0     35  0.833333    0.118443   
2   针杆藻属          Synedra sp.  硅藻     5060.0     38  0.904762    0.103689   
3  标志星杆藻  Asterionella notata  硅藻     2120.0     23  0.547619    0.043443   

      优势度_Y 是否优势种  
0  0.186661   优势种  
1  0.098702   优势种  
2  0.093813   优势种  
3  0.023790   优势种  


In [40]:
dominant_diatom = dominant_species[
    dominant_species["类群"] == "硅藻"
]

dominant_dinoflagellate = dominant_species[
    dominant_species["类群"] == "甲藻"
]

dominant_cyanobacteria = dominant_species[
    dominant_species["类群"] == "蓝藻"
]

In [41]:
print("硅藻优势种")
print(dominant_diatom)

print("甲藻优势种")
print(dominant_dinoflagellate)

print("蓝藻优势种")
print(dominant_cyanobacteria)

硅藻优势种
     中文名                  拉丁名  类群  物种总丰度_n_i  出现样品数  出现频率_f_i  相对丰度_n_i_N  \
0   菱形藻属        Nitzschia sp.  硅藻    10340.0     37  0.880952    0.211885   
1   舟形藻属         Navicula sp.  硅藻     5780.0     35  0.833333    0.118443   
2   针杆藻属          Synedra sp.  硅藻     5060.0     38  0.904762    0.103689   
3  标志星杆藻  Asterionella notata  硅藻     2120.0     23  0.547619    0.043443   

      优势度_Y 是否优势种  
0  0.186661   优势种  
1  0.098702   优势种  
2  0.093813   优势种  
3  0.023790   优势种  
甲藻优势种
Empty DataFrame
Columns: [中文名, 拉丁名, 类群, 物种总丰度_n_i, 出现样品数, 出现频率_f_i, 相对丰度_n_i_N, 优势度_Y, 是否优势种]
Index: []
蓝藻优势种
Empty DataFrame
Columns: [中文名, 拉丁名, 类群, 物种总丰度_n_i, 出现样品数, 出现频率_f_i, 相对丰度_n_i_N, 优势度_Y, 是否优势种]
Index: []


In [42]:
output_file = "浮游植物丰度和优势种结果.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(
        writer,
        sheet_name="原始数据加丰度",
        index=False
    )

    species_sample.to_excel(
        writer,
        sheet_name="样品物种丰度",
        index=False
    )

    sample_summary.to_excel(
        writer,
        sheet_name="样品类群丰度",
        index=False
    )

    species_summary.to_excel(
        writer,
        sheet_name="所有物种优势度",
        index=False
    )

    dominant_species.to_excel(
        writer,
        sheet_name="整体优势种",
        index=False
    )

    dominant_diatom.to_excel(
        writer,
        sheet_name="硅藻优势种",
        index=False
    )

    dominant_dinoflagellate.to_excel(
        writer,
        sheet_name="甲藻优势种",
        index=False
    )

    dominant_cyanobacteria.to_excel(
        writer,
        sheet_name="蓝藻优势种",
        index=False
    )

print(f"结果已经保存至：{output_file}")

结果已经保存至：浮游植物丰度和优势种结果.xlsx


In [43]:
b_data = species_sample[
    species_sample["样品编号"].str.endswith(" B", na=False)
].copy()

In [44]:
print(sorted(b_data["样品编号"].unique()))
print("B层样品数：", b_data["样品编号"].nunique())

['PT-1 B', 'PT-2 B', 'PT-3 B', 'PT-4 B', 'PT-6 B', 'XS-1 B', 'XS-10 B', 'XS-11 B', 'XS-12 B', 'XS-2 B', 'XS-3 B', 'XS-4 B', 'XS-5 B', 'XS-6 B', 'XS-7 B', 'XS-8 B', 'XS-9 B']
B层样品数： 17


In [45]:
target_groups = ["硅藻", "甲藻", "蓝藻"]

b_group_abundance = (
    b_data[
        b_data["类群"].isin(target_groups)
    ]
    .groupby(
        ["样品编号", "类群"],
        as_index=False
    )["丰度_cells_L"]
    .sum()
    .pivot(
        index="样品编号",
        columns="类群",
        values="丰度_cells_L"
    )
    .fillna(0)
    .reset_index()
)

In [46]:
for group in target_groups:
    if group not in b_group_abundance.columns:
        b_group_abundance[group] = 0

In [47]:
b_group_abundance = b_group_abundance[
    ["样品编号", "硅藻", "甲藻", "蓝藻"]
].rename(
    columns={
        "硅藻": "硅藻丰度_cells_L",
        "甲藻": "甲藻丰度_cells_L",
        "蓝藻": "蓝藻丰度_cells_L"
    }
)

In [48]:
b_total_abundance = (
    b_data.groupby(
        "样品编号",
        as_index=False
    )["丰度_cells_L"]
    .sum()
    .rename(
        columns={
            "丰度_cells_L": "浮游植物总丰度_cells_L"
        }
    )
)

In [49]:
b_sample_summary = b_group_abundance.merge(
    b_total_abundance,
    on="样品编号",
    how="left"
)

In [50]:
b_sample_summary["硅藻相对丰度"] = (
    b_sample_summary["硅藻丰度_cells_L"]
    / b_sample_summary["浮游植物总丰度_cells_L"]
)

b_sample_summary["甲藻相对丰度"] = (
    b_sample_summary["甲藻丰度_cells_L"]
    / b_sample_summary["浮游植物总丰度_cells_L"]
)

b_sample_summary["蓝藻相对丰度"] = (
    b_sample_summary["蓝藻丰度_cells_L"]
    / b_sample_summary["浮游植物总丰度_cells_L"]
)

In [51]:
b_total_samples = b_data["样品编号"].nunique()

In [53]:
b_total_samples

17

In [52]:
b_species_summary = (
    b_data.groupby(
        ["中文名", "拉丁名", "类群"],
        as_index=False
    )
    .agg(
        物种总丰度_n_i=("丰度_cells_L", "sum"),
        出现样品数=("样品编号", "nunique")
    )
)

In [54]:
b_N = b_species_summary["物种总丰度_n_i"].sum()

In [55]:
b_species_summary["出现频率_f_i"] = (
    b_species_summary["出现样品数"]
    / b_total_samples
)

In [56]:
b_species_summary["相对丰度_n_i_N"] = (
    b_species_summary["物种总丰度_n_i"]
    / b_N
)

In [57]:
b_species_summary["优势度_Y"] = (
    b_species_summary["相对丰度_n_i_N"]
    * b_species_summary["出现频率_f_i"]
)

In [58]:
b_species_summary["是否优势种"] = np.where(
    b_species_summary["优势度_Y"] > 0.02,
    "优势种",
    "非优势种"
)

In [59]:
b_dominant_species = (
    b_species_summary[
        b_species_summary["优势度_Y"] > 0.02
    ]
    .sort_values("优势度_Y", ascending=False)
    .reset_index(drop=True)
)

In [60]:
print("B层样品数：", b_total_samples)
print("B层累计总丰度：", b_N)

print(
    b_species_summary[
        [
            "中文名",
            "拉丁名",
            "类群",
            "物种总丰度_n_i",
            "出现样品数",
            "出现频率_f_i",
            "相对丰度_n_i_N",
            "优势度_Y",
            "是否优势种"
        ]
    ].head(20)
)

print("B层优势种：")
print(b_dominant_species)

B层样品数： 17
B层累计总丰度： 18260.0
          中文名                               拉丁名  类群  物种总丰度_n_i  出现样品数  \
0       三齿盒形藻                Biddulphia tridens  硅藻       40.0      2   
1       中沙角管藻         Cerataulina zhongshaensis  硅藻       20.0      1   
2   中肋双眉藻膨大变种       Amphora costata v. inflata   硅藻       40.0      2   
3       串珠梯楔藻         Climacosphenia moniligera  硅藻       60.0      2   
4        五角角藻               Ceratium pentagonum  甲藻       20.0      1   
5    亚德里亚海杆线藻             Rhabdonema adriaticum  硅藻       20.0      1   
6       伯氏根管藻             Rhizosolenia bergonii  硅藻      120.0      5   
7     具边线性圆筛藻  Coscinodiscus marginato-lineatus  硅藻      500.0      8   
8       刚毛根管藻             Rhizosolenia setigera  硅藻       40.0      2   
9        半盘藻属                    Hemidiscus sp.  硅藻       40.0      1   
10      南海条纹藻             Striatella nanhainica  硅藻       20.0      1   
11       卵形藻属                     Cocconeis sp.  硅藻       80.0      4   
12      原多甲藻属           

In [61]:
print(sorted(b_data["样品编号"].unique()))
print(b_total_samples)

['PT-1 B', 'PT-2 B', 'PT-3 B', 'PT-4 B', 'PT-6 B', 'XS-1 B', 'XS-10 B', 'XS-11 B', 'XS-12 B', 'XS-2 B', 'XS-3 B', 'XS-4 B', 'XS-5 B', 'XS-6 B', 'XS-7 B', 'XS-8 B', 'XS-9 B']
17


In [62]:
print(b_species_summary["相对丰度_n_i_N"].sum())

1.0


In [63]:
b_species_summary = (
    b_species_summary
    .sort_values("优势度_Y", ascending=False)
    .reset_index(drop=True)
)

In [64]:
print(b_species_summary.head(10))

       中文名                               拉丁名  类群  物种总丰度_n_i  出现样品数  出现频率_f_i  \
0     菱形藻属                     Nitzschia sp.  硅藻     5080.0     16  0.941176   
1     针杆藻属                       Synedra sp.  硅藻     2180.0     17  1.000000   
2     舟形藻属                      Navicula sp.  硅藻     2260.0     16  0.941176   
3    标志星杆藻               Asterionella notata  硅藻     1520.0     12  0.705882   
4     小环藻属                    Cyclotella sp.  硅藻      820.0     11  0.647059   
5     圆筛藻属                 Coscinodiscus sp.  硅藻      400.0     10  0.588235   
6  具边线性圆筛藻  Coscinodiscus marginato-lineatus  硅藻      500.0      8  0.470588   
7     粗根管藻              Rhizosolenia robusta  硅藻      300.0      9  0.529412   
8     脆杆藻属                    Fragilaria sp.  硅藻      380.0      7  0.411765   
9      羽纹藻                    Pinnularia sp.  硅藻      360.0      6  0.352941   

   相对丰度_n_i_N     优势度_Y 是否优势种  
0    0.278204  0.261839   优势种  
1    0.119387  0.119387   优势种  
2    0.123768  0.116487

In [65]:
output_file = "B层浮游植物丰度与优势种结果.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    b_data.to_excel(
        writer,
        sheet_name="B层样品物种丰度",
        index=False
    )

    b_sample_summary.to_excel(
        writer,
        sheet_name="B层类群丰度",
        index=False
    )

    b_species_summary.to_excel(
        writer,
        sheet_name="B层全部物种优势度",
        index=False
    )

    b_dominant_species.to_excel(
        writer,
        sheet_name="B层优势种",
        index=False
    )

print(f"结果已保存到：{output_file}")

结果已保存到：B层浮游植物丰度与优势种结果.xlsx


In [66]:
b_sample_list = pd.DataFrame({
    "样品编号": sorted(b_data["样品编号"].unique())
})

In [67]:
output_file = "B层浮游植物丰度与优势种结果.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    b_sample_list.to_excel(
        writer,
        sheet_name="B层样品清单",
        index=False
    )

    b_data.to_excel(
        writer,
        sheet_name="B层样品物种丰度",
        index=False
    )

    b_sample_summary.to_excel(
        writer,
        sheet_name="B层类群丰度",
        index=False
    )

    b_species_summary.to_excel(
        writer,
        sheet_name="B层全部物种优势度",
        index=False
    )

    b_dominant_species.to_excel(
        writer,
        sheet_name="B层优势种",
        index=False
    )

print(f"结果已保存到：{output_file}")

结果已保存到：B层浮游植物丰度与优势种结果.xlsx


In [68]:
# 1. 增加百分比列
b_sample_summary["硅藻相对丰度_%"] = (
    b_sample_summary["硅藻相对丰度"] * 100
)

b_sample_summary["甲藻相对丰度_%"] = (
    b_sample_summary["甲藻相对丰度"] * 100
)

b_sample_summary["蓝藻相对丰度_%"] = (
    b_sample_summary["蓝藻相对丰度"] * 100
)

b_species_summary["相对丰度_%"] = (
    b_species_summary["相对丰度_n_i_N"] * 100
)

# 2. 重新按照优势度排序
b_species_summary = (
    b_species_summary
    .sort_values("优势度_Y", ascending=False)
    .reset_index(drop=True)
)

# 3. 提取优势种
b_dominant_species = (
    b_species_summary[
        b_species_summary["优势度_Y"] > 0.02
    ]
    .copy()
)

# 4. 样品清单
b_sample_list = pd.DataFrame({
    "样品编号": sorted(b_data["样品编号"].unique())
})

# 5. 导出 Excel
output_file = "B层浮游植物丰度与优势种结果.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    b_sample_list.to_excel(
        writer,
        sheet_name="B层样品清单",
        index=False
    )

    b_data.to_excel(
        writer,
        sheet_name="B层样品物种丰度",
        index=False
    )

    b_sample_summary.to_excel(
        writer,
        sheet_name="B层类群丰度",
        index=False
    )

    b_species_summary.to_excel(
        writer,
        sheet_name="B层全部物种优势度",
        index=False
    )

    b_dominant_species.to_excel(
        writer,
        sheet_name="B层优势种",
        index=False
    )

print(f"结果已保存到：{output_file}")

结果已保存到：B层浮游植物丰度与优势种结果.xlsx
